In [1]:
"""
Resume Screening Automation
---------------------------
Pipeline:
  1. Load resumes (+ a job description)
  2. Clean the text
  3. Extract SKILLS (spaCy PhraseMatcher) and EXPERIENCE (regex)
  4. Match resumes to the job with TF-IDF + cosine similarity
  5. Combine into a final score and rank candidates

Install:  pip install spacy scikit-learn pandas
Optional: python -m spacy download en_core_web_sm   (not required here)
"""

import re
import pandas as pd
import spacy
from spacy.matcher import PhraseMatcher
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# ----------------------------------------------------------------------
# 1. DATA  (replace with your intern resume database)
# ----------------------------------------------------------------------
# Real data options:
#   df = pd.read_csv("resumes.csv")   # needs columns: name, resume_text
# For PDFs use: pip install pdfplumber, then extract text per file.
resumes = pd.DataFrame({
    "name": ["Ayesha", "Bilal", "Chen"],
    "resume_text": [
        "Data science student with 1 year of experience. Skills: Python, "
        "pandas, numpy, scikit-learn, SQL, machine learning, matplotlib.",
        "Web developer, 3 years experience in JavaScript, React, HTML, CSS, "
        "Node.js and some SQL.",
        "Final-year student. Built NLP projects using Python, spaCy, NLTK, "
        "TF-IDF, deep learning and TensorFlow. 6 months internship experience.",
    ],
})

job_description = """
Data Science Intern: looking for someone with Python, pandas, SQL,
machine learning, scikit-learn and NLP knowledge. Basic experience preferred.
"""

# ----------------------------------------------------------------------
# 2. CLEANING
# ----------------------------------------------------------------------
def clean_text(text: str) -> str:
    text = text.lower()
    text = re.sub(r"[^a-z0-9+#.\s]", " ", text)  # keep c++, c#, node.js
    return re.sub(r"\s+", " ", text).strip()

resumes["clean_text"] = resumes["resume_text"].apply(clean_text)
job_clean = clean_text(job_description)

# ----------------------------------------------------------------------
# 3a. SKILL EXTRACTION with spaCy
# ----------------------------------------------------------------------
# A skills dictionary is the simplest reliable approach. Extend it!
SKILLS = [
    "python", "sql", "pandas", "numpy", "scikit-learn", "machine learning",
    "deep learning", "nlp", "spacy", "nltk", "tensorflow", "pytorch",
    "tf-idf", "matplotlib", "javascript", "react", "html", "css", "node.js",
    "excel", "tableau", "power bi", "statistics",
]

nlp = spacy.blank("en")  # blank pipeline = no model download needed
matcher = PhraseMatcher(nlp.vocab, attr="LOWER")
matcher.add("SKILLS", [nlp.make_doc(s) for s in SKILLS])

def extract_skills(text: str) -> set:
    doc = nlp(text)
    return {doc[start:end].text.lower() for _, start, end in matcher(doc)}

resumes["skills"] = resumes["clean_text"].apply(extract_skills)
job_skills = extract_skills(job_clean)

# ----------------------------------------------------------------------
# 3b. EXPERIENCE EXTRACTION (regex)
# ----------------------------------------------------------------------
def extract_experience_years(text: str) -> float:
    """Finds phrases like '3 years', '1.5 yrs', '6 months'."""
    years = 0.0
    for num, unit in re.findall(r"(\d+(?:\.\d+)?)\s*(year|yr|month)s?", text):
        years += float(num) / (12 if unit == "month" else 1)
    return round(years, 2)

resumes["experience_years"] = resumes["clean_text"].apply(extract_experience_years)

# ----------------------------------------------------------------------
# 4. MATCHING with TF-IDF + cosine similarity
# ----------------------------------------------------------------------
# TF-IDF turns text into numbers: words that are frequent in one document
# but rare overall get higher weight. Cosine similarity then measures how
# close each resume vector is to the job vector (0 = unrelated, 1 = same).
vectorizer = TfidfVectorizer(stop_words="english", ngram_range=(1, 2))
matrix = vectorizer.fit_transform(list(resumes["clean_text"]) + [job_clean])
resumes["tfidf_score"] = cosine_similarity(matrix[:-1], matrix[-1]).ravel()

# Skill overlap: what fraction of the job's required skills does the candidate have?
def skill_overlap(cand_skills: set) -> float:
    return len(cand_skills & job_skills) / len(job_skills) if job_skills else 0.0

resumes["skill_match"] = resumes["skills"].apply(skill_overlap)
resumes["missing_skills"] = resumes["skills"].apply(lambda s: sorted(job_skills - s))

# ----------------------------------------------------------------------
# 5. FINAL SCORE + RANKING
# ----------------------------------------------------------------------
# Weights are a design choice; justify/tune them in your report.
resumes["final_score"] = 0.5 * resumes["skill_match"] + 0.5 * resumes["tfidf_score"]
ranked = resumes.sort_values("final_score", ascending=False)

print(ranked[["name", "skill_match", "tfidf_score", "experience_years",
              "final_score", "missing_skills"]].round(3).to_string(index=False))

# ranked.to_csv("ranked_candidates.csv", index=False)

# ----------------------------------------------------------------------
# OPTIONAL: BERT upgrade (understands meaning, not just shared words)
# pip install sentence-transformers
#
# from sentence_transformers import SentenceTransformer
# model = SentenceTransformer("all-MiniLM-L6-v2")
# emb = model.encode(list(resumes["clean_text"]) + [job_clean])
# resumes["bert_score"] = cosine_similarity(emb[:-1], emb[-1:]).ravel()
# ----------------------------------------------------------------------

  name  skill_match  tfidf_score  experience_years  final_score                          missing_skills
Ayesha          0.8        0.367               1.0        0.583                                   [nlp]
  Chen          0.4        0.062               0.5        0.231         [machine learning, pandas, sql]
 Bilal          0.2        0.031               3.0        0.116 [machine learning, nlp, pandas, python]
